# HyPER Light (`hyper_light`) and HyPER (`hyper`)

Short examples for the meta-prompt optimizers in `coolprompt.optimizer.hyper`.

**Requirements**
- Install CoolPrompt (editable from repo root is fine): `pip install -e ".[dev]"` or your usual env.
- A LangChain-compatible chat model. Below we use **OpenRouter** via `langchain_openai.ChatOpenAI` (`OPENROUTER_API_KEY` or `OPENAI_API_KEY`).

**Note:** `hyper` runs many model calls per iteration (paraphrases, scoring, feedback, inner meta-prompt). Start with tiny datasets and `n_iterations=1` if you are testing spend.

## 0. Environment and imports

In [1]:
import os
from langchain_openai import ChatOpenAI

from coolprompt.optimizer.autoprompting_method import build_benchmark_context
from coolprompt.optimizer.hyper.hyper import HyPERMethod
from coolprompt.optimizer.hyper.meta_prompt import HyPERLightMethod
from coolprompt.utils.var_validation import validate_method

/Users/zhuravlevvikt/coolprompt_testing/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
os.environ["OPENAI_API_KEY"] = "sk-proj-..."

In [7]:
def make_chat_model() -> ChatOpenAI:
    api_key = os.environ.get("OPENROUTER_API_KEY") or os.environ.get("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError(
            "Set OPENROUTER_API_KEY or OPENAI_API_KEY in the environment (or in a .env loaded before this cell)."
        )
    model = os.environ.get("INTEGRATION_LLM_MODEL", "gpt-4o-mini")
    return ChatOpenAI(
        model=model,
        temperature=0.3,
        max_completion_tokens=1024,
        max_retries=3,
        api_key=api_key,
    )


llm = make_chat_model()
llm

ChatOpenAI(output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x362f63c50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x362f63d90>, root_client=<openai.OpenAI object at 0x362fd4550>, root_async_client=<openai.AsyncOpenAI object at 0x362f63ed0>, model_name='gpt-4o-mini', temperature=0.3, model_kwargs={}, openai_api_

## 1. HyPER Light — one meta-prompt step (`hyper_light`)

Single LLM call: the model returns an improved prompt inside `<result_prompt>...</result_prompt>`.

In [8]:
method = HyPERLightMethod()
assert method.name == "hyper_light"

improved = method.optimize(
    model=llm,
    initial_prompt=(
        "You classify movie reviews as POSITIVE or NEGATIVE. "
        "Reply with only the label."
    ),
    problem_description="Binary sentiment on short English reviews.",
    meta_prompt_context={
        "style": "concise",
        "output_contract": "single token: POSITIVE or NEGATIVE",
    },
)

print(improved[:2000])


# Role
You are an AI trained to classify sentiments in text, specifically focusing on movie reviews.

# Task context
The task involves determining the sentiment of short English movie reviews and categorizing them as either POSITIVE or NEGATIVE.

# Instructions
- Analyze the provided movie review.
- Classify the sentiment as either POSITIVE or NEGATIVE based on the content of the review.
- Respond with only the label.

# Output requirements
- The response must be a single token: either POSITIVE or NEGATIVE.



### Registered name (same as `PromptTuner` / benchmarks)

In [9]:
impl = validate_method("hyper_light")
impl.name, type(impl).__name__

('hyper_light', 'HyPERLightMethod')

## 2. HyPER — outer loop with mini-batch scoring (`hyper`)

Loads **GSM8K** the same way as YAML benchmarks: `build_benchmark_context` + `dataset.configuration` **`20/20/1`** → **20 train / 20 val** from the train split (40 rows, 50% validation) plus a 1-example test slice. Metric: **exact match (em)**.

The same `llm` runs paraphrases, feedback, inner meta-prompt, and batched answer generation in the evaluator. **BERTScore** (MMR) loads on first use — can take a minute; keep `n_iterations` small when experimenting.

In [10]:
benchmark_config = {
    "dataset": {"name": "gsm8k", "configuration": "30/50/1"},
    "task": "generation",
    "metric": "em",
}

ctx = build_benchmark_context(llm, benchmark_config)
train_x, val_x, train_y, val_y = ctx.dataset_split
print(f"GSM8K split — train={len(train_x)}  val={len(val_x)}  test_slice={len(ctx.test_dataset)}")

hyper = HyPERMethod()
assert hyper.name == "hyper"

starter = (
    "solve the problem "
)

final_prompt = hyper.optimize(
    model=llm,
    initial_prompt=starter,
    dataset_split=ctx.dataset_split,
    evaluator=ctx.evaluator,
    problem_description="math solving",
    meta_prompt_context={
        "problem_description": "math solving"
    },
    n_iterations=1,
    patience=None,
    n_candidates=3,
    top_n_candidates=3,
    k_samples=3,
    mini_batch_size=8,
    contrastive_probability=0.0,
    enable_instance_leak_audit=False,
    random_seed=42,
)

print("--- final prompt ---")
print(final_prompt)

[2026-06-05 23:57:00,904] [INFO] [evaluator.__init__] - Evaluator successfully initialized with em metric
[2026-06-05 23:57:00,908] [INFO] [hyper.optimize] - [HyPER] Starting optimization
[2026-06-05 23:57:00,908] [DEBUG] [hyper.optimize] - [HyPER] Initial prompt: solve the problem ...
[2026-06-05 23:57:00,908] [INFO] [evaluator.evaluate] - Evaluating prompt for generation task on 50 samples


GSM8K split — train=30  val=50  test_slice=1


Evaluating: 100%|██████████| 50/50 [00:22<00:00,  2.22sample/s]
[2026-06-05 23:57:23,410] [INFO] [hyper.optimize] - [HyPER] Initial validation score: 0.9800
[2026-06-05 23:57:23,411] [INFO] [hyper.optimize] - [HyPER] Config: n_iterations=1, patience=None, n_candidates=3, mini_batch_size=8, k_samples=3, top_n_candidates=3
HyPER iterations:   0%|          | 0/1 [00:00<?, ?it/s][2026-06-05 23:57:23,413] [INFO] [hyper.optimize] - 
[2026-06-05 23:57:23,413] [INFO] [hyper.optimize] - [HyPER] === Iteration 1/1 | best_score=0.9800 ===
[2026-06-05 23:57:23,414] [DEBUG] [hyper.optimize] - [HyPER] Current best prompt: solve the problem ...
[2026-06-05 23:57:23,414] [INFO] [hyper.optimize] - [HyPER] Step 1: Generating 3 paraphrase candidates...
[2026-06-05 23:57:25,350] [INFO] [hyper.optimize] - [HyPER] Generated 4 candidates (including original)
[2026-06-05 23:57:25,352] [INFO] [hyper.optimize] - [HyPER] Step 2: Mini-batch sampled: 8 examples (seed=42, indices=[20, 3, 0, 23, 8, 7, 24, 4])
[2026-0

--- final prompt ---
solve the problem 
